# Lightweight Machine Learning-Based Network Intrusion Detection for Resource-Constrained Edge Devices

## Project Overview

This notebook implements and evaluates a lightweight machine learning-based Network Intrusion Detection System (NIDS) using the Edge-IIoTset dataset.

### Research Objective
To evaluate the trade-off between network attack detection performance and computational resource consumption when using lightweight machine learning approaches.

### Dataset
Edge-IIoTset:
- ML-EdgeIIoT-dataset.csv
- Individual attack and normal traffic CSV files where required.

### Evaluation Environment
- Hardware: MacBook Pro
- Language: Python
- Libraries: Pandas, NumPy, Scikit-learn and Matplotlib.

### Research Scope
The study focuses on controlled host-based evaluation as a proxy for resource-constrained edge deployment. It does not claim physical edge-device validation.

In [3]:
import os
import sys
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-13.7.8-x86_64-i386-64bit
Pandas: 3.0.2
NumPy: 1.26.4
Scikit-learn: 1.5.1


## 1.1 Dataset Configuration

The study uses the Edge-IIoTset dataset. The main dataset selected for machine learning experiments is ML-EdgeIIoT-dataset.csv.

The individual attack and normal traffic CSV files will only be examined where necessary for dataset verification and evaluation design.

In [4]:
# Define the dataset location
DATASET_PATH = os.path.join(
    "Edge-IIoTset dataset",
    "Selected dataset for ML and DL",
    "ML-EdgeIIoT-dataset.csv"
)

# Verify that the dataset exists
if os.path.exists(DATASET_PATH):
    print("Dataset found:", DATASET_PATH)
    print("File size:", round(os.path.getsize(DATASET_PATH) / (1024 * 1024), 2), "MB")
else:
    print("Dataset not found. Check the project directory.")

Dataset found: Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv
File size: 78.38 MB


## 2. Dataset Inspection

This section examines the structure of the selected Edge-IIoTset dataset before any preprocessing or model training.

The inspection focuses on:
- Dataset dimensions.
- Feature names and data types.
- Target labels.
- Potential capture or session identifiers.
- Initial sample records.

In [5]:
# Load a small sample for initial inspection
sample_df = pd.read_csv(DATASET_PATH, nrows=5)

# Display dataset structure
print("Number of columns:", len(sample_df.columns))

print("\nColumn names:")
for column in sample_df.columns:
    print("-", column)

print("\nData types:")
display(sample_df.dtypes.to_frame(name="Data Type"))

print("\nFirst five records:")
display(sample_df)

Number of columns: 63

Column names:
- frame.time
- ip.src_host
- ip.dst_host
- arp.dst.proto_ipv4
- arp.opcode
- arp.hw.size
- arp.src.proto_ipv4
- icmp.checksum
- icmp.seq_le
- icmp.transmit_timestamp
- icmp.unused
- http.file_data
- http.content_length
- http.request.uri.query
- http.request.method
- http.referer
- http.request.full_uri
- http.request.version
- http.response
- http.tls_port
- tcp.ack
- tcp.ack_raw
- tcp.checksum
- tcp.connection.fin
- tcp.connection.rst
- tcp.connection.syn
- tcp.connection.synack
- tcp.dstport
- tcp.flags
- tcp.flags.ack
- tcp.len
- tcp.options
- tcp.payload
- tcp.seq
- tcp.srcport
- udp.port
- udp.stream
- udp.time_delta
- dns.qry.name
- dns.qry.name.len
- dns.qry.qu
- dns.qry.type
- dns.retransmission
- dns.retransmit_request
- dns.retransmit_request_in
- mqtt.conack.flags
- mqtt.conflag.cleansess
- mqtt.conflags
- mqtt.hdrflags
- mqtt.len
- mqtt.msg_decoded_as
- mqtt.msg
- mqtt.msgtype
- mqtt.proto_len
- mqtt.protoname
- mqtt.topic
- mqtt.topic_

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64



First five records:


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [6]:
# Count records using the CSV file structure
with open(DATASET_PATH, "r", encoding="utf-8") as file:
    total_rows = sum(1 for _ in file) - 1

print("Total records:", f"{total_rows:,}")
print("Total features:", len(sample_df.columns))

Total records: 157,800
Total features: 63


## 3. Target Variable and Data Type Inspection

This section examines the target variables, `Attack_label` and `Attack_type`, to understand how normal and malicious traffic is represented. It also checks the data types of the dataset fields before preprocessing.

In [7]:
# Inspect data types
display(sample_df.dtypes.to_frame(name="Data Type"))

# Load only the target columns
target_df = pd.read_csv(
    DATASET_PATH,
    usecols=["Attack_label", "Attack_type"]
)

# Examine target values
print("Attack_label distribution:")
display(target_df["Attack_label"].value_counts(dropna=False))

print("\nAttack_type distribution:")
display(target_df["Attack_type"].value_counts(dropna=False))

# Check missing target values
print("\nMissing target values:")
display(target_df[["Attack_label", "Attack_type"]].isnull().sum())

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64


Attack_label distribution:


Attack_label
1    133499
0     24301
Name: count, dtype: int64


Attack_type distribution:


Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64


Missing target values:


Attack_label    0
Attack_type     0
dtype: int64

In [8]:
# Load the two target columns
target_df = pd.read_csv(
    DATASET_PATH,
    usecols=["Attack_label", "Attack_type"]
)

print("Attack_label distribution:")
display(
    target_df["Attack_label"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("Attack_type distribution:")
display(
    target_df["Attack_type"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("Missing target values:")
display(target_df.isnull().sum())

Attack_label distribution:


,Count
Attack_label,
1,133499
0,24301


Attack_type distribution:


,Count
Attack_type,
Normal,24301
DDoS_UDP,14498
DDoS_ICMP,14090
Ransomware,10925
DDoS_HTTP,10561
SQL_injection,10311
Uploading,10269
DDoS_TCP,10247
Backdoor,10195


Missing target values:


Attack_label    0
Attack_type     0
dtype: int64

## 4. Data Quality Assessment

This section examines missing values, duplicate records and the general quality of the dataset. The results will help identify data preparation requirements before model development.

In [9]:
# Load the approved dataset
df = pd.read_csv(DATASET_PATH)

# Dataset dimensions
print("Dataset shape:", df.shape)

# Missing values
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": (missing_values / len(df)) * 100
})

display(
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Values", ascending=False)
)

# Duplicate records
duplicate_count = df.duplicated().sum()

print("Duplicate records:", f"{duplicate_count:,}")
print(
    "Duplicate percentage:",
    f"{(duplicate_count / len(df)) * 100:.2f}%"
)

/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1121213083.py:2: DtypeWarning: Columns (0: arp.dst.proto_ipv4, 1: arp.src.proto_ipv4, 2: http.file_data, 3: http.request.uri.query, 4: http.request.method, 5: http.referer, 6: http.request.full_uri, 7: http.request.version, 8: tcp.options, 9: tcp.payload, 10: tcp.srcport, 11: dns.qry.name.len, 12: mqtt.conack.flags, 13: mqtt.msg, 14: mqtt.protoname, 15: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH)


Dataset shape: (157800, 63)


,Missing Values,Missing Percentage (%)


Duplicate records: 814
Duplicate percentage: 0.52%


In [10]:
# Confirm missing values across all columns
print("Total missing values:", df.isnull().sum().sum())

# Show columns containing missing values
missing_columns = df.isnull().sum()
display(missing_columns[missing_columns > 0])

# Inspect duplicate records
duplicate_rows = df[df.duplicated(keep=False)]

print("Rows involved in duplication:", len(duplicate_rows))
display(duplicate_rows.head(10))

Total missing values: 0


Series([], dtype: int64)

Rows involved in duplication: 821


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
5,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
6,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
7,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
8,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
9,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


## 5. Feature and Capture Identifier Assessment

This section examines the available network traffic fields to identify potential identifiers that could support capture/session-grouped evaluation. It also distinguishes network metadata from potential predictive features.

In [11]:
# Display all columns with their data types
feature_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique(dropna=False).values
})

display(feature_info)

# Inspect columns that may contain capture or session information
identifier_keywords = [
    "capture", "session", "flow", "stream",
    "file", "time", "id"
]

candidate_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in identifier_keywords)
]

print("Potential identifier columns:")
for col in candidate_columns:
    print(f"{col}: {df[col].nunique(dropna=False):,} unique values")

,Column,Data Type,Unique Values
0,frame.time,str,155186
1,ip.src_host,str,19090
2,ip.dst_host,str,8084
3,arp.dst.proto_ipv4,object,9
4,arp.opcode,float64,3
...,...,...,...
58,mbtcp.len,float64,1
59,mbtcp.trans_id,float64,1
60,mbtcp.unit_id,float64,1
61,Attack_label,int64,2


Potential identifier columns:
frame.time: 155,186 unique values
icmp.transmit_timestamp: 84 unique values
http.file_data: 497 unique values
udp.stream: 14,492 unique values
udp.time_delta: 39 unique values
mbtcp.trans_id: 1 unique values
mbtcp.unit_id: 1 unique values


In [12]:
from pathlib import Path

# Locate the approved dataset directory
dataset_root = Path("Edge-IIoTset dataset")

print("Dataset files available for capture-level investigation:\n")

for file in sorted(dataset_root.rglob("*.csv")):
    if file.name == "ML-EdgeIIoT-dataset.csv":
        continue

    print(f"{file.name} | {file.stat().st_size / (1024**2):.2f} MB")

Dataset files available for capture-level investigation:

Backdoor_attack.csv | 17.06 MB
DDoS_HTTP_Flood_attack.csv | 91.28 MB
DDoS_ICMP_Flood_attack.csv | 850.75 MB
DDoS_TCP_SYN_Flood_attack.csv | 897.59 MB
DDoS_UDP_Flood_attack.csv | 858.53 MB
MITM_attack.csv | 0.31 MB
OS_Fingerprinting_attack.csv | 0.30 MB
Password_attack.csv | 623.03 MB
Port_Scanning_attack.csv | 7.00 MB
Ransomware_attack.csv | 7.24 MB
SQL_injection_attack.csv | 25.69 MB
Uploading_attack.csv | 16.70 MB
Vulnerability_scanner_attack.csv | 167.61 MB
XSS_attack.csv | 6.64 MB
Distance.csv | 349.10 MB
Flame_Sensor.csv | 327.61 MB
Heart_Rate.csv | 50.13 MB
IR_Receiver.csv | 393.62 MB
Modbus.csv | 64.75 MB
Soil_Moisture.csv | 363.91 MB
Sound_Sensor.csv | 456.78 MB
Temperature_and_Humidity.csv | 879.85 MB
Water_Level.csv | 697.57 MB
phValue.csv | 226.33 MB
DNN-EdgeIIoT-dataset.csv | 1161.02 MB


In [13]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

files_to_inspect = [
    "MITM_attack.csv",
    "OS_Fingerprinting_attack.csv",
    "Backdoor_attack.csv",
    "Port_Scanning_attack.csv",
    "Distance.csv",
    "Heart_Rate.csv"
]

for filename in files_to_inspect:
    file_path = next(dataset_root.rglob(filename))

    print(f"\n{'=' * 60}")
    print("FILE:", filename)

    sample = pd.read_csv(file_path, nrows=5)

    print("Columns:", len(sample.columns))
    print("Column names:")
    print(sample.columns.tolist())

    print("\nFirst five records:")
    display(sample.head())


FILE: MITM_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgt

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM



FILE: OS_Fingerprinting_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 22:14:30.939803000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x00002ea2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
1,2021 22:14:30.939896000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0x000036a2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
2,2021 22:14:31.371846000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x0000058a,256.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
3,2021 22:14:31.371927000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0x00000d8a,256.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
4,2021 22:14:31.910480000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x0000d3ea,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting



FILE: Backdoor_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 17:58:23.150209000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
1,2021 17:58:23.150231000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
2,2021 17:58:23.166306000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
3,2021 17:58:23.269052000,0,0,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
4,2021 17:58:23.659941000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor


... (2 outputs left out)


FILE: Distance.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgtype

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 23:58:21.314757000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 23:58:21.314960000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
2,2021 23:58:21.315192000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
3,2021 23:58:21.315258000,192.168.1.128,192.168.1.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 23:58:21.315294000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal



FILE: Heart_Rate.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgty

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 02:39:39.986982000,192.168.3.18,192.168.3.12,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 02:39:39.987068000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
2,2021 02:39:40.000498000,192.168.3.18,192.168.3.12,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
3,2021 02:39:40.000575000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 02:39:40.001196000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal


In [14]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

capture_files = [
    file for file in dataset_root.rglob("*.csv")
    if file.name != "ML-EdgeIIoT-dataset.csv"
    and file.name != "DNN-EdgeIIoT-dataset.csv"
]

results = []

for file in sorted(capture_files):

    # Read only the target columns
    sample = pd.read_csv(
        file,
        usecols=["Attack_label", "Attack_type"]
    )

    results.append({
        "File": file.name,
        "Records": len(sample),
        "Attack Types": ", ".join(
            sample["Attack_type"].dropna().unique().astype(str)
        ),
        "Labels": ", ".join(
            sample["Attack_label"].dropna().unique().astype(str)
        )
    })

capture_summary = pd.DataFrame(results)

display(capture_summary)

,File,Records,Attack Types,Labels
0,Backdoor_attack.csv,24862,Backdoor,1
1,DDoS_HTTP_Flood_attack.csv,229022,DDoS_HTTP,1
2,DDoS_ICMP_Flood_attack.csv,2914354,DDoS_ICMP,1
3,DDoS_TCP_SYN_Flood_attack.csv,2020120,DDoS_TCP,1
4,DDoS_UDP_Flood_attack.csv,3201626,DDoS_UDP,1
5,MITM_attack.csv,1229,MITM,1
6,OS_Fingerprinting_attack.csv,1001,OS_Fingerprinting,1
7,Password_attack.csv,1053385,Password,1
8,Port_Scanning_attack.csv,22564,Port_Scanning,1
9,Ransomware_attack.csv,10925,Ransomware,1


In [15]:
# Compare selected ML dataset counts with full capture-file counts

capture_counts = capture_summary[
    ["File", "Records", "Attack Types"]
].copy()

# Standardise the one known naming difference
capture_counts["Attack Types"] = capture_counts["Attack Types"].replace(
    {"OS_Fingerprinting": "Fingerprinting"}
)

capture_counts = (
    capture_counts.groupby("Attack Types")["Records"]
    .sum()
    .rename("Capture File Records")
)

ml_counts = (
    df["Attack_type"]
    .value_counts()
    .rename("Selected ML Records")
)

comparison = pd.concat(
    [capture_counts, ml_counts],
    axis=1
).fillna(0)

comparison["Selected ML Records"] = (
    comparison["Selected ML Records"].astype(int)
)

comparison["Capture File Records"] = (
    comparison["Capture File Records"].astype(int)
)

comparison["ML Dataset Coverage (%)"] = (
    comparison["Selected ML Records"]
    / comparison["Capture File Records"] * 100
).round(2)

display(comparison.sort_values("Selected ML Records", ascending=False))

,Capture File Records,Selected ML Records,ML Dataset Coverage (%)
Normal,11209913,24301,0.22
DDoS_UDP,3201626,14498,0.45
DDoS_ICMP,2914354,14090,0.48
Ransomware,10925,10925,100.00
DDoS_HTTP,229022,10561,4.61
SQL_injection,51203,10311,20.14
Uploading,37634,10269,27.29
DDoS_TCP,2020120,10247,0.51
Backdoor,24862,10195,41.01
Vulnerability_scanner,145869,10076,6.91


In [16]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

# Use several fields to identify sampled records
check_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "Attack_label",
    "Attack_type"
]

# Map selected dataset labels to their capture filenames
capture_mapping = {
    "Backdoor": "Backdoor_attack.csv",
    "DDoS_HTTP": "DDoS_HTTP_Flood_attack.csv",
    "DDoS_ICMP": "DDoS_ICMP_Flood_attack.csv",
    "DDoS_TCP": "DDoS_TCP_SYN_Flood_attack.csv",
    "DDoS_UDP": "DDoS_UDP_Flood_attack.csv",
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv",
    "Password": "Password_attack.csv",
    "Port_Scanning": "Port_Scanning_attack.csv",
    "Ransomware": "Ransomware_attack.csv",
    "SQL_injection": "SQL_injection_attack.csv",
    "Uploading": "Uploading_attack.csv",
    "Vulnerability_scanner": "Vulnerability_scanner_attack.csv",
    "XSS": "XSS_attack.csv"
}

normal_files = [
    "Distance.csv", "Flame_Sensor.csv", "Heart_Rate.csv",
    "IR_Receiver.csv", "Modbus.csv", "Soil_Moisture.csv",
    "Sound_Sensor.csv", "Temperature_and_Humidity.csv",
    "Water_Level.csv", "phValue.csv"
]

# Select up to five records per category
samples = {}

for category in df["Attack_type"].unique():
    samples[category] = df.loc[
        df["Attack_type"] == category, check_columns
    ].sample(
        n=min(5, (df["Attack_type"] == category).sum()),
        random_state=42
    )

# Convert values to comparable strings
def make_keys(data):
    return set(
        pd.util.hash_pandas_object(
            data[check_columns].astype("string").fillna("<NA>"),
            index=False
        ).tolist()
    )

results = []

for category, sample in samples.items():

    if category == "Normal":
        files_to_check = normal_files
    else:
        files_to_check = [capture_mapping[category]]

    sample_keys = make_keys(sample)
    matched_keys = set()

    for filename in files_to_check:
        file_path = next(dataset_root.rglob(filename))

        for chunk in pd.read_csv(
            file_path,
            usecols=check_columns,
            chunksize=100_000
        ):
            matched_keys.update(
                make_keys(chunk).intersection(sample_keys)
            )

    results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched Records": len(matched_keys)
    })

match_results = pd.DataFrame(results)
display(match_results)

/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns

,Attack Type,Sample Records,Matched Records
0,MITM,5,0
1,Fingerprinting,5,0
2,Ransomware,5,5
3,Uploading,5,5
4,SQL_injection,5,5
5,DDoS_HTTP,5,5
6,DDoS_TCP,5,5
7,Password,5,5
8,Port_Scanning,5,5
9,Vulnerability_scanner,5,5


In [20]:
def make_keys(data, columns):
    return set(
        pd.util.hash_pandas_object(
            data[columns].astype("string").fillna("<NA>"),
            index=False
        ).tolist()
    )

In [21]:
diagnostic_columns = [
    col for col in df.columns
    if col not in ["frame.time", "Attack_type"]
]

diagnostic_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

diagnostic_results = []

for category, filename in diagnostic_mapping.items():

    sample = df.loc[
        df["Attack_type"] == category,
        diagnostic_columns
    ].sample(n=5, random_state=42)

    sample_keys = make_keys(sample, diagnostic_columns)

    matched_keys = set()
    file_path = next(dataset_root.rglob(filename))

    for chunk in pd.read_csv(
        file_path,
        usecols=diagnostic_columns,
        chunksize=100_000
    ):
        chunk_keys = make_keys(chunk, diagnostic_columns)
        matched_keys.update(chunk_keys.intersection(sample_keys))

    diagnostic_results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched Records": len(matched_keys)
    })

display(pd.DataFrame(diagnostic_results))

,Attack Type,Sample Records,Matched Records
0,MITM,5,0
1,Fingerprinting,5,0


In [23]:
# Inspect selected ML records against their original capture files

inspection_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "udp.stream",
    "Attack_label",
    "Attack_type"
]

inspection_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

for category, filename in inspection_mapping.items():

    print(f"\n{'='*60}")
    print(f"ATTACK TYPE: {category}")
    print(f"{'='*60}")

    selected_sample = df.loc[
        df["Attack_type"] == category,
        inspection_columns
    ].sample(n=3, random_state=42)

    capture_path = next(dataset_root.rglob(filename))

    capture_data = pd.read_csv(
        capture_path,
        usecols=inspection_columns
    )

    print("\nSELECTED ML DATASET — SAMPLE:")
    display(selected_sample)

    print("\nCAPTURE FILE — FIRST THREE RECORDS:")
    display(capture_data[inspection_columns].head(3))

    print("\nDATA TYPES — SELECTED DATASET:")
    print(df[inspection_columns].dtypes)

    print("\nDATA TYPES — CAPTURE FILE:")
    print(capture_data.dtypes)


ATTACK TYPE: MITM

SELECTED ML DATASET — SAMPLE:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
739,0.0,0,0.0,DESKTOP-UHF0SF2.local,0.0,0.0,1,MITM
788,0.0,0,0.0,0,0.0,0.0,1,MITM
43,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,MITM



CAPTURE FILE — FIRST THREE RECORDS:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
0,0.0,0,0.0,_ipps._tcp.local,0,16.0,1,MITM
1,6.0,192.168.0.152,0.0,0,0,0.0,1,MITM
2,6.0,192.168.0.101,0.0,0,0,0.0,1,MITM



DATA TYPES — SELECTED DATASET:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.srcport      object
tcp.dstport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

DATA TYPES — CAPTURE FILE:
frame.time      float64
ip.src_host         str
ip.dst_host     float64
tcp.dstport         str
tcp.srcport         str
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

ATTACK TYPE: Fingerprinting

SELECTED ML DATASET — SAMPLE:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
1735,2021 22:22:32.191112000,192.168.0.128,192.168.0.170,65535.0,4683.0,0.0,1,Fingerprinting
2155,2021 22:27:37.774518000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,Fingerprinting
1955,2021 22:25:20.622133000,192.168.0.128,192.168.0.170,65535.0,43567.0,0.0,1,Fingerprinting



CAPTURE FILE — FIRST THREE RECORDS:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
0,2021 22:14:30.939803000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,OS_Fingerprinting
1,2021 22:14:30.939896000,192.168.0.128,192.168.0.170,0.0,0.0,0.0,1,OS_Fingerprinting
2,2021 22:14:31.371846000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,OS_Fingerprinting



DATA TYPES — SELECTED DATASET:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.srcport      object
tcp.dstport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

DATA TYPES — CAPTURE FILE:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.dstport     float64
tcp.srcport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object


In [24]:
from decimal import Decimal, InvalidOperation

def normalise_value(value):
    if pd.isna(value):
        return "<NA>"

    value = str(value).strip()

    try:
        number = Decimal(value)
        if number.is_finite():
            return str(number.normalize())
    except InvalidOperation:
        pass

    return value


def normalised_keys(data, columns):
    normalised = data[columns].apply(
        lambda column: column.map(normalise_value)
    )

    return set(
        pd.util.hash_pandas_object(
            normalised,
            index=False
        )
    )


comparison_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "udp.stream",
    "Attack_label"
]

diagnostic_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

normalised_results = []

for category, filename in diagnostic_mapping.items():

    sample = df.loc[
        df["Attack_type"] == category,
        comparison_columns
    ].sample(n=5, random_state=42)

    sample_keys = normalised_keys(sample, comparison_columns)

    capture_path = next(dataset_root.rglob(filename))
    matched_keys = set()

    for chunk in pd.read_csv(
        capture_path,
        usecols=comparison_columns,
        chunksize=100_000
    ):
        chunk_keys = normalised_keys(chunk, comparison_columns)
        matched_keys.update(chunk_keys.intersection(sample_keys))

    normalised_results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched After Normalisation": len(matched_keys)
    })

display(pd.DataFrame(normalised_results))

,Attack Type,Sample Records,Matched After Normalisation
0,MITM,5,4
1,Fingerprinting,5,5


In [25]:
# Identify unmatched MITM sample records

category = "MITM"
filename = "MITM_attack.csv"

sample = df.loc[
    df["Attack_type"] == category,
    comparison_columns
].sample(n=5, random_state=42).copy()

def normalised_tuples(data, columns):
    normalised = data[columns].apply(
        lambda column: column.map(normalise_value)
    )
    return set(map(tuple, normalised.to_numpy()))

sample_tuples = normalised_tuples(sample, comparison_columns)

capture_path = next(dataset_root.rglob(filename))
capture_tuples = set()

for chunk in pd.read_csv(
    capture_path,
    usecols=comparison_columns,
    chunksize=100_000
):
    capture_tuples.update(
        normalised_tuples(chunk, comparison_columns)
    )

unmatched_mask = [
    row not in capture_tuples
    for row in sample_tuples
]

# Preserve the original sample row order
sample["Match Status"] = [
    "Unmatched" if tuple(
        normalise_value(value) for value in row
    ) not in capture_tuples else "Matched"
    for row in sample[comparison_columns].to_numpy()
]

display(sample)

,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Match Status
739,0.0,0,0.0,DESKTOP-UHF0SF2.local,0.0,0.0,1,Matched
788,0.0,0,0.0,0,0.0,0.0,1,Matched
43,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,Matched
155,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,Matched
494,0.0,0,0.0,_googlecast._tcp.local,0.0,0.0,1,Matched


In [26]:
# Final assessment of candidate grouping fields

group_candidates = [
    "udp.stream",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport"
]

group_assessment = []

for column in group_candidates:
    group_assessment.append({
        "Feature": column,
        "Data Type": str(df[column].dtype),
        "Unique Values": df[column].nunique(),
        "Missing Values": df[column].isna().sum(),
        "Unique Value Ratio (%)": round(
            df[column].nunique() / len(df) * 100, 2
        )
    })

display(pd.DataFrame(group_assessment))

,Feature,Data Type,Unique Values,Missing Values,Unique Value Ratio (%)
0,udp.stream,float64,14492,0,9.18
1,ip.src_host,str,19090,0,12.10
2,ip.dst_host,str,8084,0,5.12
3,tcp.srcport,object,33551,0,21.26
4,tcp.dstport,float64,23188,0,14.69


In [27]:
# Section 3: Initial feature preparation

# Preserve the original dataframe
model_df = df.copy()

# Define classification targets
binary_target = "Attack_label"
multiclass_target = "Attack_type"

# Separate input features from target variables
X = model_df.drop(
    columns=[binary_target, multiclass_target]
)

y_binary = model_df[binary_target]
y_multiclass = model_df[multiclass_target]

# Identify numerical and categorical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Dataset shape:", X.shape)
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nBinary target shape:", y_binary.shape)
print("Multiclass target shape:", y_multiclass.shape)

Dataset shape: (157800, 61)
Numerical features: 42
Categorical features: 19

Numerical features:
['arp.opcode', 'arp.hw.size', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.content_length', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.seq', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']

Categorical features:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.reques

In [28]:
# Inspect categorical and identifier-like features

feature_cardinality = pd.DataFrame({
    "Feature": X.columns,
    "Data Type": X.dtypes.astype(str).values,
    "Unique Values": [X[col].nunique() for col in X.columns],
    "Unique Ratio (%)": [
        round(X[col].nunique() / len(X) * 100, 2)
        for col in X.columns
    ]
})

display(
    feature_cardinality.sort_values(
        by="Unique Values",
        ascending=False
    ).reset_index(drop=True)
)

,Feature,Data Type,Unique Values,Unique Ratio (%)
0,frame.time,str,155186,98.34
1,tcp.ack_raw,float64,94716,60.02
2,tcp.options,object,73140,46.35
3,tcp.checksum,float64,55513,35.18
4,tcp.srcport,object,33551,21.26
...,...,...,...,...
56,mqtt.msg_decoded_as,float64,1,0.00
57,icmp.unused,float64,1,0.00
58,mbtcp.len,float64,1,0.00
59,mbtcp.trans_id,float64,1,0.00


In [29]:
# Initial feature reduction

# Identify constant features
constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

# Exclude fields that are unsuitable for the initial lightweight model
excluded_features = [
    "frame.time",
    "tcp.ack_raw",
    "tcp.checksum",
    "tcp.options",
    "tcp.payload",
    "http.file_data",
    "http.request.uri.query",
    "http.referer",
    "http.request.full_uri",
    "mqtt.msg",
    "ip.src_host",
    "ip.dst_host",
    "udp.stream"
]

# Combine exclusion lists without duplicates
features_to_remove = list(
    dict.fromkeys(constant_features + excluded_features)
)

# Keep only features that exist in X
features_to_remove = [
    col for col in features_to_remove if col in X.columns
]

# Create reduced feature dataset
X_reduced = X.drop(columns=features_to_remove)

print("Original feature count:", X.shape[1])
print("Removed feature count:", len(features_to_remove))
print("Remaining feature count:", X_reduced.shape[1])

print("\nRemoved features:")
print(features_to_remove)

print("\nRemaining dataset shape:", X_reduced.shape)

Original feature count: 61
Removed feature count: 21
Remaining feature count: 40

Removed features:
['icmp.unused', 'http.tls_port', 'dns.qry.type', 'dns.retransmit_request_in', 'mqtt.msg_decoded_as', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id', 'frame.time', 'tcp.ack_raw', 'tcp.checksum', 'tcp.options', 'tcp.payload', 'http.file_data', 'http.request.uri.query', 'http.referer', 'http.request.full_uri', 'mqtt.msg', 'ip.src_host', 'ip.dst_host', 'udp.stream']

Remaining dataset shape: (157800, 40)


In [30]:
# Identify feature types after feature reduction

numerical_features = X_reduced.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_reduced.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Reduced dataset shape:", X_reduced.shape)
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Reduced dataset shape: (157800, 40)
Numerical features: 31
Categorical features: 9

Categorical features:
['arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.request.method', 'http.request.version', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic']

Numerical features:
['arp.opcode', 'arp.hw.size', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'http.content_length', 'http.response', 'tcp.ack', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.seq', 'udp.port', 'udp.time_delta', 'dns.qry.name', 'dns.qry.qu', 'dns.retransmission', 'dns.retransmit_request', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver']


## Splitting the binary classification data

In [34]:
from sklearn.model_selection import train_test_split

# Primary classification task: binary intrusion detection
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (126240, 40)
Testing features: (31560, 40)

Training target distribution:
Attack_label
1    0.846
0    0.154
Name: proportion, dtype: float64

Testing target distribution:
Attack_label
1    0.846
0    0.154
Name: proportion, dtype: float64


## Building the preprocessing pipeline

In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        lambda data: data.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Fit on training data and transform both partitions
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

print("Training matrix shape:", X_train_processed.shape)
print("Testing matrix shape:", X_test_processed.shape)

print("Training matrix type:", type(X_train_processed))
print("Training matrix data type:", X_train_processed.dtype)

Preprocessing completed successfully.
Training matrix shape: (126240, 28240)
Testing matrix shape: (31560, 28240)
Training matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Training matrix data type: float64


## Establishing a baseline

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
import time

# Initialise baseline classifier
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

baseline_model.fit(X_train_processed, y_train)

training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred = baseline_model.predict(X_test_processed)
y_prob = baseline_model.predict_proba(X_test_processed)[:, 1]

prediction_time = time.perf_counter() - start_time

# Evaluate classification performance
print("BASELINE LOGISTIC REGRESSION RESULTS")
print("------------------------------------")

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nTraining time (seconds):", round(training_time, 4))
print("Prediction time (seconds):", round(prediction_time, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

BASELINE LOGISTIC REGRESSION RESULTS
------------------------------------
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
ROC-AUC: 1.0

Training time (seconds): 1.8589
Prediction time (seconds): 0.0134

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4860
           1       1.00      1.00      1.00     26700

    accuracy                           1.00     31560
   macro avg       1.00      1.00      1.00     31560
weighted avg       1.00      1.00      1.00     31560



In [38]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# 1. Inspect the strongest Logistic Regression coefficients
feature_names = preprocessor.get_feature_names_out()
coefficients = baseline_model.coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
})

print("TOP 20 FEATURES BY ABSOLUTE COEFFICIENT")
display(
    coefficient_table.sort_values(
        "Absolute Coefficient",
        ascending=False
    ).head(20)
)

# 2. Check for identical predictor rows across train and test
train_hashes = set(
    pd.util.hash_pandas_object(X_train, index=False)
)

test_hashes = set(
    pd.util.hash_pandas_object(X_test, index=False)
)

overlapping_rows = len(train_hashes.intersection(test_hashes))

print("\nTRAIN-TEST OVERLAP CHECK")
print("Unique training rows:", len(train_hashes))
print("Unique testing rows:", len(test_hashes))
print("Identical rows in both partitions:", overlapping_rows)

# 3. Display the confusion matrix
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

TOP 20 FEATURES BY ABSOLUTE COEFFICIENT


,Feature,Coefficient,Absolute Coefficient
28235,cat__mqtt.protoname_0.0,2.472651,2.472651
28238,cat__mqtt.topic_0.0,2.472651,2.472651
28232,cat__mqtt.conack.flags_0.0,2.472651,2.472651
28234,cat__mqtt.protoname_0,-2.344483,2.344483
28237,cat__mqtt.topic_0,-2.342358,2.342358
28231,cat__mqtt.conack.flags_0,-2.251130,2.251130
28224,cat__dns.qry.name.len_0.0,0.882873,0.882873
28223,cat__dns.qry.name.len_0,-0.769204,0.769204
2594,cat__tcp.srcport_1883.0,-0.743924,0.743924
12208,cat__tcp.srcport_4321.0,0.519838,0.519838



TRAIN-TEST OVERLAP CHECK
Unique training rows: 119360
Unique testing rows: 30526
Identical rows in both partitions: 1637

CONFUSION MATRIX
[[ 4860     0]
 [    0 26700]]


In [39]:
# Remove duplicate predictor rows from training data only

train_data = X_train.copy()
train_data["target"] = y_train

print("Training rows before:", len(train_data))

train_data = train_data.drop_duplicates(
    subset=X_train.columns,
    keep="first"
)

print("Training rows after:", len(train_data))
print("Training duplicates removed:",
      len(X_train) - len(train_data))

# Recreate training features and target
X_train_unique = train_data.drop(columns=["target"])
y_train_unique = train_data["target"]

# Fit a fresh preprocessing pipeline
preprocessor_unique = preprocessor.__class__(
    transformers=preprocessor.transformers
)

X_train_unique_processed = preprocessor_unique.fit_transform(
    X_train_unique
)

X_test_unique_processed = preprocessor_unique.transform(X_test)

# Train a fresh Logistic Regression model
model_unique = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_unique.fit(
    X_train_unique_processed,
    y_train_unique
)

# Evaluate on the original test set
y_pred_unique = model_unique.predict(X_test_unique_processed)

print("\nRESULTS AFTER TRAINING DEDUPLICATION")
print("-----------------------------------")

print("Accuracy:",
      round(accuracy_score(y_test, y_pred_unique), 4))

print("Precision:",
      round(precision_score(y_test, y_pred_unique), 4))

print("Recall:",
      round(recall_score(y_test, y_pred_unique), 4))

print("F1-score:",
      round(f1_score(y_test, y_pred_unique), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_unique))

Training rows before: 126240
Training rows after: 120238
Training duplicates removed: 6002

RESULTS AFTER TRAINING DEDUPLICATION
-----------------------------------
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Confusion matrix:
[[ 4860     0]
 [    0 26700]]


In [40]:
# Check whether identical predictor rows have conflicting target labels

duplicate_label_check = (
    model_df
    .groupby(list(X_reduced.columns), dropna=False)["Attack_label"]
    .nunique()
)

print("Unique predictor groups:", len(duplicate_label_check))
print("Groups with conflicting labels:", (duplicate_label_check > 1).sum())
print("Groups with consistent labels:", (duplicate_label_check == 1).sum())

Unique predictor groups: 149559
Groups with conflicting labels: 0
Groups with consistent labels: 149559


In [41]:
# Measure how strongly individual numerical features relate to Attack_label

feature_correlations = (
    X_reduced[numerical_features]
    .corrwith(y_binary)
    .abs()
    .sort_values(ascending=False)
)

print("Top 15 numerical features associated with Attack_label:")
display(feature_correlations.head(15))

Top 15 numerical features associated with Attack_label:


mqtt.msgtype              0.292345
mqtt.hdrflags             0.292345
mqtt.len                  0.272520
tcp.seq                   0.269419
tcp.dstport               0.242699
mqtt.ver                  0.209438
mqtt.conflag.cleansess    0.209438
mqtt.conflags             0.209438
mqtt.proto_len            0.209438
mqtt.topic_len            0.209100
tcp.flags.ack             0.186849
tcp.flags                 0.171915
icmp.seq_le               0.121194
icmp.checksum             0.116663
tcp.connection.syn        0.101786
dtype: float64

In [42]:
# Examine attack distribution across categorical feature values

for col in categorical_features:
    print(f"\nFeature: {col}")

    distribution = pd.crosstab(
        X_reduced[col],
        y_binary,
        normalize="index"
    )

    distribution.columns = ["Normal (%)", "Attack (%)"]
    distribution = distribution * 100

    display(distribution.sort_values("Attack (%)", ascending=False).head(10))


Feature: arp.dst.proto_ipv4


,Normal (%),Attack (%)
arp.dst.proto_ipv4,,
0,0.000000,100.000000
0.0,0.000000,100.000000
192.168.0.170,0.000000,100.000000
192.168.0.128,0.341131,99.658869
192.168.0.1,4.597701,95.402299
192.168.0.147,9.865471,90.134529
192.168.0.101,16.666667,83.333333
0,19.119339,80.880661
192.168.0.129,100.000000,0.000000



Feature: arp.src.proto_ipv4


,Normal (%),Attack (%)
arp.src.proto_ipv4,,
0,0.000000,100.000000
0.0,0.000000,100.000000
192.168.0.170,0.000000,100.000000
192.168.0.128,0.146413,99.853587
192.168.0.101,6.250000,93.750000
192.168.0.1,12.218650,87.781350
0,22.045620,77.954380
0.0.0.0,100.000000,0.000000
192.168.0.129,100.000000,0.000000



Feature: http.request.method


,Normal (%),Attack (%)
http.request.method,,
0,0.000000,100.000000
GET,0.000000,100.000000
OPTIONS,0.000000,100.000000
POST,0.000000,100.000000
TRACE,0.000000,100.000000
0.0,18.298709,81.701291
0.0,33.688996,66.311004


... (8 outputs left out)


Feature: mqtt.protoname


,Normal (%),Attack (%)
mqtt.protoname,,
0.0,0.0,100.0
0.0,0.0,100.0
0,100.0,0.0
MQTT,100.0,0.0



Feature: mqtt.topic


,Normal (%),Attack (%)
mqtt.topic,,
0.0,0.0,100.0
0.0,0.0,100.0
0,100.0,0.0
Temperature_and_Humidity,100.0,0.0


In [43]:
# Diagnostic experiment: exclude strongly class-associated categorical fields

suspicious_categorical = [
    "arp.dst.proto_ipv4",
    "arp.src.proto_ipv4",
    "http.request.method",
    "http.request.version",
    "mqtt.conack.flags",
    "mqtt.protoname",
    "mqtt.topic"
]

X_diagnostic = X_reduced.drop(columns=suspicious_categorical)

# Recreate the train/test split using the same random seed
X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(
    X_diagnostic,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Identify remaining feature types
numerical_diag = X_train_diag.select_dtypes(include=["number"]).columns.tolist()
categorical_diag = X_train_diag.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
numerical_transformer_diag = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_diag = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        lambda data: data.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_diag = ColumnTransformer(transformers=[
    ("num", numerical_transformer_diag, numerical_diag),
    ("cat", categorical_transformer_diag, categorical_diag)
])

X_train_diag_processed = preprocessor_diag.fit_transform(X_train_diag)
X_test_diag_processed = preprocessor_diag.transform(X_test_diag)

# Train Logistic Regression
model_diag = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_diag.fit(X_train_diag_processed, y_train_diag)

# Evaluate
y_pred_diag = model_diag.predict(X_test_diag_processed)

print("RESULTS WITHOUT SUSPICIOUS CATEGORICAL FEATURES")
print("-----------------------------------------------")
print("Accuracy:", accuracy_score(y_test_diag, y_pred_diag))
print("Precision:", precision_score(y_test_diag, y_pred_diag))
print("Recall:", recall_score(y_test_diag, y_pred_diag))
print("F1-score:", f1_score(y_test_diag, y_pred_diag))
print("\nConfusion matrix:")
print(confusion_matrix(y_test_diag, y_pred_diag))

RESULTS WITHOUT SUSPICIOUS CATEGORICAL FEATURES
-----------------------------------------------
Accuracy: 0.9848859315589353
Precision: 0.9861332542360313
Recall: 0.9961423220973783
F1-score: 0.9911125188649363

Confusion matrix:
[[ 4486   374]
 [  103 26597]]


In [44]:
# Examine the remaining numerical features by target class

for col in numerical_diag:
    print(f"\nFeature: {col}")

    display(
        X_train_diag.assign(Attack_label=y_train_diag)
        .groupby("Attack_label")[col]
        .agg(["mean", "median", "std", "min", "max"])
        .round(4)
    )


Feature: arp.opcode


,mean,median,std,min,max
Attack_label,,,,,
0,0.0022,0.0,0.0496,0.0,2.0
1,0.0164,0.0,0.1614,0.0,2.0



Feature: arp.hw.size


,mean,median,std,min,max
Attack_label,,,,,
0,0.012,0.0,0.2685,0.0,6.0
1,0.069,0.0,0.6399,0.0,6.0



Feature: icmp.checksum


,mean,median,std,min,max
Attack_label,,,,,
0,0.0000,0.0,0.0000,0.0,0.0
1,3619.3672,0.0,12058.7296,0.0,65532.0


... (52 outputs left out)


Feature: mqtt.topic_len


,mean,median,std,min,max
Attack_label,,,,,
0,1.2394,0.0,5.3115,0.0,24.0
1,0.0000,0.0,0.0000,0.0,0.0



Feature: mqtt.ver


,mean,median,std,min,max
Attack_label,,,,,
0,0.2016,0.0,0.8752,0.0,4.0
1,0.0000,0.0,0.0000,0.0,0.0


In [45]:
# Diagnostic experiment: remove selected protocol-specific indicators

protocol_indicators = [
    "icmp.transmit_timestamp",
    "http.content_length",
    "http.response",
    "tcp.connection.synack",
    "tcp.connection.fin",
    "tcp.connection.rst",
    "dns.retransmission",
    "dns.retransmit_request"
]

X_general = X_diagnostic.drop(columns=protocol_indicators)

# Split data
X_train_general, X_test_general, y_train_general, y_test_general = train_test_split(
    X_general,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Identify feature types
numerical_general = X_train_general.select_dtypes(include=["number"]).columns.tolist()
categorical_general = X_train_general.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
preprocessor_general = ColumnTransformer(transformers=[
    ("num", numerical_transformer_diag, numerical_general),
    ("cat", categorical_transformer_diag, categorical_general)
])

X_train_general_processed = preprocessor_general.fit_transform(X_train_general)
X_test_general_processed = preprocessor_general.transform(X_test_general)

# Train model
model_general = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_general.fit(X_train_general_processed, y_train_general)

# Evaluate
y_pred_general = model_general.predict(X_test_general_processed)

print("RESULTS USING GENERAL NETWORK FEATURES")
print("--------------------------------------")
print("Accuracy:", accuracy_score(y_test_general, y_pred_general))
print("Precision:", precision_score(y_test_general, y_pred_general))
print("Recall:", recall_score(y_test_general, y_pred_general))
print("F1-score:", f1_score(y_test_general, y_pred_general))
print("\nConfusion matrix:")
print(confusion_matrix(y_test_general, y_pred_general))

RESULTS USING GENERAL NETWORK FEATURES
--------------------------------------
Accuracy: 0.985044359949303
Precision: 0.9860637509266124
Recall: 0.9964044943820225
F1-score: 0.9912071535022354

Confusion matrix:
[[ 4484   376]
 [   96 26604]]


In [46]:
# Retain only numerical features from the general feature set
numerical_general = [
    col for col in numerical_features
    if col in X_general.columns
]

X_numerical = X_general[numerical_general]

# Use the same train/test split
X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(
    X_numerical,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# Numerical preprocessing
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_num_processed = numerical_pipeline.fit_transform(X_train_num)
X_test_num_processed = numerical_pipeline.transform(X_test_num)

# Train Logistic Regression
numerical_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

numerical_model.fit(X_train_num_processed, y_train_num)

# Evaluate
y_pred_num = numerical_model.predict(X_test_num_processed)

print("RESULTS USING NUMERICAL FEATURES ONLY")
print("-------------------------------------")
print("Accuracy:", accuracy_score(y_test_num, y_pred_num))
print("Precision:", precision_score(y_test_num, y_pred_num))
print("Recall:", recall_score(y_test_num, y_pred_num))
print("F1-score:", f1_score(y_test_num, y_pred_num))
print("\nConfusion matrix:")
print(confusion_matrix(y_test_num, y_pred_num))

RESULTS USING NUMERICAL FEATURES ONLY
-------------------------------------
Accuracy: 0.7536755386565273
Precision: 0.9300581712415924
Recall: 0.7664794007490636
F1-score: 0.8403827201051248

Confusion matrix:
[[ 3321  1539]
 [ 6235 20465]]


In [47]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import time

# Use the general feature set
X_tree = X_general.copy()

# Identify numerical and categorical columns
tree_numerical = [
    col for col in numerical_features
    if col in X_tree.columns
]

tree_categorical = [
    col for col in categorical_features
    if col in X_tree.columns
]

# Preprocess numerical features
tree_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Preprocess categorical features
tree_cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        lambda data: data.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", tree_num_pipeline, tree_numerical),
        ("cat", tree_cat_pipeline, tree_categorical)
    ]
)

# Use the same training and testing split
X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# Transform data
X_train_tree_processed = tree_preprocessor.fit_transform(X_train_tree)
X_test_tree_processed = tree_preprocessor.transform(X_test_tree)

# Initialise model
tree_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

tree_model.fit(X_train_tree_processed, y_train_tree)

training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred_tree = tree_model.predict(X_test_tree_processed)

prediction_time = time.perf_counter() - start_time

# Evaluate model
print("DECISION TREE RESULTS")
print("---------------------")
print("Accuracy:", accuracy_score(y_test_tree, y_pred_tree))
print("Precision:", precision_score(y_test_tree, y_pred_tree))
print("Recall:", recall_score(y_test_tree, y_pred_tree))
print("F1-score:", f1_score(y_test_tree, y_pred_tree))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_tree, y_pred_tree))

print("\nTraining time:", round(training_time, 4), "seconds")
print("Prediction time:", round(prediction_time, 4), "seconds")

DECISION TREE RESULTS
---------------------
Accuracy: 0.9999049429657795
Precision: 0.9998876530726885
Recall: 1.0
F1-score: 0.999943823380709

Confusion matrix:
[[ 4857     3]
 [    0 26700]]

Training time: 0.8206 seconds
Prediction time: 0.0058 seconds


In [48]:
import numpy as np
import pandas as pd
import joblib
from io import BytesIO

# Decision Tree structural complexity
print("DECISION TREE COMPLEXITY")
print("------------------------")
print("Tree depth:", tree_model.get_depth())
print("Number of leaves:", tree_model.get_n_leaves())
print("Total nodes:", tree_model.tree_.node_count)

# Estimate serialized model size
model_buffer = BytesIO()
joblib.dump(tree_model, model_buffer)

model_size_mb = len(model_buffer.getvalue()) / (1024 ** 2)
print("Serialized model size:", round(model_size_mb, 4), "MB")

# Identify the most influential features
feature_names = tree_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTOP 15 IMPORTANT FEATURES")
display(importance_df.head(15))

DECISION TREE COMPLEXITY
------------------------
Tree depth: 10
Number of leaves: 21
Total nodes: 41
Serialized model size: 0.0044 MB

TOP 15 IMPORTANT FEATURES


,Feature,Importance
28189,cat__dns.qry.name.len_0,4.223462e-01
6,num__tcp.dstport,1.660902e-01
2560,cat__tcp.srcport_1883.0,1.613865e-01
22608,cat__tcp.srcport_5900.0,1.095892e-01
17,num__mqtt.hdrflags,8.044810e-02
24162,cat__tcp.srcport_60944.0,5.761182e-02
11,num__udp.port,2.255425e-03
4,num__tcp.ack,1.312163e-04
10,num__tcp.seq,5.623612e-05
3,num__icmp.seq_le,4.332769e-05


In [49]:
# Identify the six dominant transformed features
dominant_features = [
    "dns.qry.name.len",
    "tcp.dstport",
    "tcp.srcport",
    "mqtt.hdrflags"
]

# Remove them from the original general feature set
X_reduced_tree = X_general.drop(
    columns=dominant_features
)

print("Remaining features:", X_reduced_tree.shape[1])

# Identify numerical and categorical columns
remaining_numerical = [
    col for col in numerical_features
    if col in X_reduced_tree.columns
]

remaining_categorical = [
    col for col in categorical_features
    if col in X_reduced_tree.columns
]

# Preprocessing
reduced_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), remaining_numerical),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("to_string", FunctionTransformer(
                lambda data: data.astype(str),
                feature_names_out="one-to-one"
            )),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), remaining_categorical)
    ]
)

# Same stratified split
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced_tree,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# Fit preprocessing
X_train_reduced_processed = reduced_preprocessor.fit_transform(
    X_train_reduced
)

X_test_reduced_processed = reduced_preprocessor.transform(
    X_test_reduced
)

# Train Decision Tree
reduced_tree_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

reduced_tree_model.fit(
    X_train_reduced_processed,
    y_train_reduced
)

# Evaluate
y_pred_reduced = reduced_tree_model.predict(
    X_test_reduced_processed
)

print("\nDECISION TREE WITHOUT DOMINANT FEATURES")
print("---------------------------------------")
print("Accuracy:", accuracy_score(y_test_reduced, y_pred_reduced))
print("Precision:", precision_score(y_test_reduced, y_pred_reduced))
print("Recall:", recall_score(y_test_reduced, y_pred_reduced))
print("F1-score:", f1_score(y_test_reduced, y_pred_reduced))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_reduced, y_pred_reduced))

Remaining features: 21

DECISION TREE WITHOUT DOMINANT FEATURES
---------------------------------------
Accuracy: 0.9468314321926489
Precision: 0.9894757433489828
Recall: 0.9472284644194756
F1-score: 0.9678913126674321

Confusion matrix:
[[ 4591   269]
 [ 1409 25291]]


In [50]:
# Inspect the candidate UDP stream identifier
print("UDP stream identifier")
print("---------------------")

print("Data type:", df["udp.stream"].dtype)
print("Missing values:", df["udp.stream"].isna().sum())
print("Unique values:", df["udp.stream"].nunique())

print("\nMost frequent values:")
display(df["udp.stream"].value_counts(dropna=False).head(15))

print("\nUnique stream counts by target:")
display(
    df.groupby("Attack_label")["udp.stream"]
      .nunique()
)

UDP stream identifier
---------------------
Data type: float64
Missing values: 0
Unique values: 14492

Most frequent values:


udp.stream
0.0        143264
53.0           21
123.0           3
5353.0          2
32770.0         2
13438.0         2
22302.0         2
29792.0         2
40704.0         2
46191.0         2
50508.0         2
55546.0         2
59348.0         2
49447.0         2
37851.0         2
Name: count, dtype: int64


Unique stream counts by target:


Attack_label
0       15
1    14478
Name: udp.stream, dtype: int64

In [51]:
# Search for potentially relevant capture/session identifiers
identifier_candidates = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "capture",
        "session",
        "stream",
        "flow",
        "trace",
        "file",
        "timestamp",
        "time"
    ])
]

print("Potential capture/session-related columns:")
print(identifier_candidates)

print("\nTheir data types and unique counts:")

display(pd.DataFrame({
    "Column": identifier_candidates,
    "Data type": [str(df[col].dtype) for col in identifier_candidates],
    "Unique values": [df[col].nunique() for col in identifier_candidates],
    "Missing values": [df[col].isna().sum() for col in identifier_candidates]
}))

Potential capture/session-related columns:
['frame.time', 'icmp.transmit_timestamp', 'http.file_data', 'udp.stream', 'udp.time_delta']

Their data types and unique counts:


,Column,Data type,Unique values,Missing values
0,frame.time,str,155186,0
1,icmp.transmit_timestamp,float64,84,0
2,http.file_data,object,497,0
3,udp.stream,float64,14492,0
4,udp.time_delta,float64,39,0


In [52]:
# Examine attack-type distribution
attack_type_counts = (
    df["Attack_type"]
    .value_counts()
    .rename_axis("Attack_type")
    .reset_index(name="Observations")
)

attack_type_counts["Percentage"] = (
    attack_type_counts["Observations"]
    / len(df) * 100
)

display(attack_type_counts)

,Attack_type,Observations,Percentage
0,Normal,24301,15.399873
1,DDoS_UDP,14498,9.187579
2,DDoS_ICMP,14090,8.929024
3,Ransomware,10925,6.923321
4,DDoS_HTTP,10561,6.692649
5,SQL_injection,10311,6.534221
6,Uploading,10269,6.507605
7,DDoS_TCP,10247,6.493663
8,Backdoor,10195,6.460710
9,Vulnerability_scanner,10076,6.385298


In [53]:
# Attack categories to hold out
held_out_attacks = [
    "MITM",
    "Fingerprinting"
]

# Separate training and testing observations
train_mask = ~df["Attack_type"].isin(held_out_attacks)
test_mask = (
    (df["Attack_type"] == "Normal") |
    df["Attack_type"].isin(held_out_attacks)
)

# Display distributions
print("PROPOSED TRAINING DISTRIBUTION")
print("--------------------------------")
display(
    df.loc[train_mask, "Attack_type"].value_counts()
)

print("\nPROPOSED TESTING DISTRIBUTION")
print("-----------------------------")
display(
    df.loc[test_mask, "Attack_type"].value_counts()
)

print("\nObservation counts")
print("Training:", train_mask.sum())
print("Testing:", test_mask.sum())

PROPOSED TRAINING DISTRIBUTION
--------------------------------


Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
Name: count, dtype: int64


PROPOSED TESTING DISTRIBUTION
-----------------------------


Attack_type
Normal            24301
MITM               1214
Fingerprinting     1001
Name: count, dtype: int64


Observation counts
Training: 155585
Testing: 26516


In [54]:
print("Current dataset shape:", df.shape)
print("Total rows:", len(df))

print("\nTraining mask:", train_mask.sum())
print("Testing mask:", test_mask.sum())
print("Overlap:", (train_mask & test_mask).sum())
print("Unassigned:", (~(train_mask | test_mask)).sum())

print("\nAttack type totals:")
display(df["Attack_type"].value_counts())

Current dataset shape: (157800, 63)
Total rows: 157800

Training mask: 155585
Testing mask: 26516
Overlap: 24301
Unassigned: 0

Attack type totals:


Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64

In [55]:
from sklearn.model_selection import train_test_split

held_out_attacks = ["MITM", "Fingerprinting"]

# Separate normal observations
normal_indices = df.index[df["Attack_type"] == "Normal"]

# Split normal observations into training and testing
normal_train_idx, normal_test_idx = train_test_split(
    normal_indices,
    test_size=0.2,
    random_state=42
)

# All attack observations except held-out categories
known_attack_mask = (
    (df["Attack_label"] == 1) &
    (~df["Attack_type"].isin(held_out_attacks))
)

# All observations belonging to held-out attack categories
held_out_mask = df["Attack_type"].isin(held_out_attacks)

# Construct final indices
train_indices = list(normal_train_idx) + list(df.index[known_attack_mask])
test_indices = list(normal_test_idx) + list(df.index[held_out_mask])

# Verify partition
print("Training observations:", len(train_indices))
print("Testing observations:", len(test_indices))
print("Total observations:", len(train_indices) + len(test_indices))

print(
    "Overlap:",
    len(set(train_indices).intersection(set(test_indices)))
)

print("\nTraining distribution:")
display(df.loc[train_indices, "Attack_type"].value_counts())

print("\nTesting distribution:")
display(df.loc[test_indices, "Attack_type"].value_counts())

Training observations: 150724
Testing observations: 7076
Total observations: 157800
Overlap: 0

Training distribution:


Attack_type
Normal                   19440
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
Name: count, dtype: int64


Testing distribution:


Attack_type
Normal            4861
MITM              1214
Fingerprinting    1001
Name: count, dtype: int64

In [58]:
from sklearn.base import clone

# Prepare training and testing data
X_train_held = X_general.loc[train_indices]
X_test_held = X_general.loc[test_indices]

y_train_held = y_binary.loc[train_indices]
y_test_held = y_binary.loc[test_indices]

# Fit preprocessing using training data only
held_preprocessor = clone(tree_preprocessor)

X_train_held_processed = held_preprocessor.fit_transform(
    X_train_held
)

X_test_held_processed = held_preprocessor.transform(
    X_test_held
)

# Initialise Decision Tree
held_tree_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

held_tree_model.fit(
    X_train_held_processed,
    y_train_held
)

held_training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred_held = held_tree_model.predict(
    X_test_held_processed
)

held_prediction_time = time.perf_counter() - start_time

# Evaluate
print("HELD-OUT ATTACK-TYPE EVALUATION")
print("--------------------------------")
print("Accuracy:", accuracy_score(y_test_held, y_pred_held))
print("Precision:", precision_score(y_test_held, y_pred_held))
print("Recall:", recall_score(y_test_held, y_pred_held))
print("F1-score:", f1_score(y_test_held, y_pred_held))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_held, y_pred_held))

print("\nTraining time:", round(held_training_time, 4), "seconds")
print("Prediction time:", round(held_prediction_time, 4), "seconds")

HELD-OUT ATTACK-TYPE EVALUATION
--------------------------------
Accuracy: 0.9672131147540983
Precision: 0.997491219267436
Recall: 0.8975169300225734
F1-score: 0.9448669201520913

Confusion matrix:
[[4856    5]
 [ 227 1988]]

Training time: 0.6074 seconds
Prediction time: 0.0014 seconds


In [59]:
from sklearn.metrics import classification_report

# Identify actual attack categories in the test partition
held_test_attack_types = df.loc[test_indices, "Attack_type"]

# Combine predictions with actual labels
held_results = pd.DataFrame({
    "Actual_Attack_Type": held_test_attack_types,
    "Actual_Binary": y_test_held,
    "Predicted_Binary": y_pred_held
})

# Evaluate each held-out category independently
print("PERFORMANCE BY HELD-OUT ATTACK CATEGORY")
print("---------------------------------------")

for attack in ["MITM", "Fingerprinting"]:

    attack_data = held_results[
        held_results["Actual_Attack_Type"] == attack
    ]

    detected = (attack_data["Predicted_Binary"] == 1).sum()
    missed = (attack_data["Predicted_Binary"] == 0).sum()

    recall = detected / len(attack_data)

    print(f"\n{attack}")
    print("Total observations:", len(attack_data))
    print("Detected:", detected)
    print("Missed:", missed)
    print("Detection recall:", round(recall, 4))

PERFORMANCE BY HELD-OUT ATTACK CATEGORY
---------------------------------------

MITM
Total observations: 1214
Detected: 987
Missed: 227
Detection recall: 0.813

Fingerprinting
Total observations: 1001
Detected: 1001
Missed: 0
Detection recall: 1.0


In [60]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import time

# Prepare the held-out training and testing data
X_train_lr = X_general.loc[train_indices]
X_test_lr = X_general.loc[test_indices]

y_train_lr = y_binary.loc[train_indices]
y_test_lr = y_binary.loc[test_indices]

# Fit preprocessing using training data only
lr_preprocessor = clone(tree_preprocessor)

X_train_lr_processed = lr_preprocessor.fit_transform(X_train_lr)
X_test_lr_processed = lr_preprocessor.transform(X_test_lr)

# Initialise Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

lr_model.fit(X_train_lr_processed, y_train_lr)

lr_training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred_lr = lr_model.predict(X_test_lr_processed)

lr_prediction_time = time.perf_counter() - start_time

# Evaluate performance
print("HELD-OUT LOGISTIC REGRESSION RESULTS")
print("------------------------------------")

print("Accuracy:", round(accuracy_score(y_test_lr, y_pred_lr), 4))
print("Precision:", round(precision_score(y_test_lr, y_pred_lr), 4))
print("Recall:", round(recall_score(y_test_lr, y_pred_lr), 4))
print("F1-score:", round(f1_score(y_test_lr, y_pred_lr), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_lr, y_pred_lr))

print("\nTraining time:", round(lr_training_time, 4), "seconds")
print("Prediction time:", round(lr_prediction_time, 4), "seconds")

HELD-OUT LOGISTIC REGRESSION RESULTS
------------------------------------
Accuracy: 0.7771
Precision: 0.5945
Recall: 0.9056
F1-score: 0.7178

Confusion Matrix:
[[3493 1368]
 [ 209 2006]]

Training time: 11.1866 seconds
Prediction time: 0.0013 seconds


In [61]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import time

# Reuse the same held-out training and testing data
X_train_rf = X_general.loc[train_indices]
X_test_rf = X_general.loc[test_indices]

y_train_rf = y_binary.loc[train_indices]
y_test_rf = y_binary.loc[test_indices]

# Fit preprocessing on training data only
rf_preprocessor = clone(tree_preprocessor)

X_train_rf_processed = rf_preprocessor.fit_transform(X_train_rf)
X_test_rf_processed = rf_preprocessor.transform(X_test_rf)

# Initialise Random Forest
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Measure training time
start_time = time.perf_counter()

rf_model.fit(X_train_rf_processed, y_train_rf)

rf_training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred_rf = rf_model.predict(X_test_rf_processed)

rf_prediction_time = time.perf_counter() - start_time

# Evaluate performance
print("HELD-OUT RANDOM FOREST RESULTS")
print("--------------------------------")

print("Accuracy:", round(accuracy_score(y_test_rf, y_pred_rf), 4))
print("Precision:", round(precision_score(y_test_rf, y_pred_rf), 4))
print("Recall:", round(recall_score(y_test_rf, y_pred_rf), 4))
print("F1-score:", round(f1_score(y_test_rf, y_pred_rf), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_rf, y_pred_rf))

print("\nTraining time:", round(rf_training_time, 4), "seconds")
print("Prediction time:", round(rf_prediction_time, 4), "seconds")

HELD-OUT RANDOM FOREST RESULTS
--------------------------------
Accuracy: 0.8185
Precision: 0.6529
Recall: 0.8975
F1-score: 0.7559

Confusion Matrix:
[[3804 1057]
 [ 227 1988]]

Training time: 2.0563 seconds
Prediction time: 0.0278 seconds


In [62]:
import joblib
import os
import tempfile

models = {
    "Logistic Regression": lr_model,
    "Decision Tree": held_tree_model,
    "Random Forest": rf_model
}

print("MODEL SERIALIZED SIZES")
print("----------------------")

for name, model in models.items():

    with tempfile.NamedTemporaryFile(suffix=".joblib") as temp_file:

        joblib.dump(model, temp_file.name)

        size_mb = os.path.getsize(temp_file.name) / (1024 * 1024)

        print(f"{name}: {size_mb:.4f} MB")

MODEL SERIALIZED SIZES
----------------------
Logistic Regression: 0.2411 MB
Decision Tree: 0.0036 MB
Random Forest: 0.1158 MB


In [65]:
def convert_to_string(data):
    return data.astype(str)

In [66]:

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        convert_to_string,
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

serializable_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [68]:
# Identify numerical and categorical columns in the actual model input
numerical_features_general = X_general.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_general = X_general.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:", len(numerical_features_general))
print("Categorical features:", len(categorical_features_general))
print("Total features:", X_general.shape[1])

Numerical features: 23
Categorical features: 2
Total features: 25


In [69]:
serializable_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features_general),
        ("cat", categorical_transformer, categorical_features_general)
    ]
)

In [70]:
from sklearn.base import clone

lr_preprocessor = clone(serializable_preprocessor)
lr_preprocessor.fit(X_train_lr)

held_preprocessor = clone(serializable_preprocessor)
held_preprocessor.fit(X_train_held)

rf_preprocessor = clone(serializable_preprocessor)
rf_preprocessor.fit(X_train_rf)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['arp.opcode', 'arp.hw.size', 'icmp.checksum',
                                  'icmp.seq_le', 'tcp.ack',
                                  'tcp.connection.syn', 'tcp.dstport',
                                  'tcp.flags', 'tcp.flags.ack', 'tcp.len',
                                  'tcp.seq', 'udp.port', 'udp.time_delta',
                                  'dns.qry.name', 'dns.qry.qu',
                                  'mqtt.conflag...
                                  'mqtt.hdrflags', 'mqtt.len', 'mqtt.msgtype',
                                  'mqtt.proto_len', 'mqtt.topic_len',
                                  'mqtt.ver']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('to_string',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<function convert_to_string at 0x1757b63e0>)),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['tcp.srcport', 'dns.qry.name.len'])])

In [71]:
from sklearn.base import clone

lr_preprocessor = clone(serializable_preprocessor)
lr_preprocessor.fit(X_train_lr)

held_preprocessor = clone(serializable_preprocessor)
held_preprocessor.fit(X_train_held)

rf_preprocessor = clone(serializable_preprocessor)
rf_preprocessor.fit(X_train_rf)

print("All three preprocessing components fitted successfully.")

All three preprocessing components fitted successfully.


In [72]:
from sklearn.pipeline import Pipeline

complete_models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", lr_preprocessor),
        ("classifier", lr_model)
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", held_preprocessor),
        ("classifier", held_tree_model)
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", rf_preprocessor),
        ("classifier", rf_model)
    ])
}

print("Complete model pipelines assembled successfully.")

Complete model pipelines assembled successfully.


In [73]:
import joblib
import os
import tempfile

print("COMPLETE PIPELINE SIZES")
print("-----------------------")

for name, pipeline in complete_models.items():
    with tempfile.NamedTemporaryFile(suffix=".joblib") as temp_file:
        joblib.dump(pipeline, temp_file.name)

        size_mb = os.path.getsize(temp_file.name) / (1024 * 1024)

        print(f"{name}: {size_mb:.4f} MB")

COMPLETE PIPELINE SIZES
-----------------------
Logistic Regression: 2.7677 MB
Decision Tree: 2.5302 MB
Random Forest: 2.6463 MB


In [74]:
import numpy as np

verification_cases = [
    (
        "Logistic Regression",
        complete_models["Logistic Regression"],
        X_test_lr,
        y_pred_lr
    ),
    (
        "Decision Tree",
        complete_models["Decision Tree"],
        X_test_held,
        y_pred_held
    ),
    (
        "Random Forest",
        complete_models["Random Forest"],
        X_test_rf,
        y_pred_rf
    )
]

print("PIPELINE PREDICTION VERIFICATION")
print("--------------------------------")

for name, pipeline, X_eval, original_predictions in verification_cases:
    pipeline_predictions = pipeline.predict(X_eval)

    identical = np.array_equal(
        pipeline_predictions,
        original_predictions
    )

    print(f"{name}: Predictions identical = {identical}")

PIPELINE PREDICTION VERIFICATION
--------------------------------
Logistic Regression: Predictions identical = False
Decision Tree: Predictions identical = False
Random Forest: Predictions identical = False


In [75]:
print("AVAILABLE PREPROCESSING OBJECTS")
print("--------------------------------")

for name, obj in list(globals().items()):
    if "preprocessor" in name.lower():
        print(f"\n{name}: {type(obj).__name__}")

        if hasattr(obj, "feature_names_in_"):
            print("Input features:", len(obj.feature_names_in_))

        if hasattr(obj, "transformers_"):
            print("Fitted transformers:", [
                item[0] for item in obj.transformers_
            ])

AVAILABLE PREPROCESSING OBJECTS
--------------------------------

preprocessor: ColumnTransformer
Input features: 40
Fitted transformers: ['num', 'cat']

preprocessor_unique: ColumnTransformer
Input features: 40
Fitted transformers: ['num', 'cat']

preprocessor_diag: ColumnTransformer
Input features: 33
Fitted transformers: ['num', 'cat']

preprocessor_general: ColumnTransformer
Input features: 25
Fitted transformers: ['num', 'cat']

tree_preprocessor: ColumnTransformer
Input features: 25
Fitted transformers: ['num', 'cat']

reduced_preprocessor: ColumnTransformer
Input features: 21
Fitted transformers: ['num', 'cat']

held_preprocessor: ColumnTransformer
Input features: 25
Fitted transformers: ['num', 'cat']

lr_preprocessor: ColumnTransformer
Input features: 25
Fitted transformers: ['num', 'cat']

rf_preprocessor: ColumnTransformer
Input features: 25
Fitted transformers: ['num', 'cat']

serializable_preprocessor: ColumnTransformer


In [76]:
import numpy as np

original_candidates = {
    "preprocessor": preprocessor,
    "preprocessor_unique": preprocessor_unique,
    "preprocessor_diag": preprocessor_diag,
    "preprocessor_general": preprocessor_general,
    "tree_preprocessor": tree_preprocessor,
    "reduced_preprocessor": reduced_preprocessor
}

model_tests = [
    ("Logistic Regression", lr_model, X_test_lr, y_pred_lr),
    ("Decision Tree", held_tree_model, X_test_held, y_pred_held),
    ("Random Forest", rf_model, X_test_rf, y_pred_rf)
]

print("ORIGINAL PREPROCESSOR COMPATIBILITY")
print("----------------------------------")

for model_name, model, X_eval, original_predictions in model_tests:
    print(f"\n{model_name}")

    for prep_name, prep in original_candidates.items():
        try:
            transformed = prep.transform(X_eval)

            if transformed.shape[1] != model.n_features_in_:
                print(
                    f"{prep_name}: incompatible dimensions "
                    f"({transformed.shape[1]} vs {model.n_features_in_})"
                )
                continue

            predictions = model.predict(transformed)

            identical = np.array_equal(
                predictions,
                original_predictions
            )

            print(f"{prep_name}: predictions identical = {identical}")

        except Exception as error:
            print(f"{prep_name}: incompatible ({type(error).__name__})")

ORIGINAL PREPROCESSOR COMPATIBILITY
----------------------------------

Logistic Regression
preprocessor: incompatible (ValueError)
preprocessor_unique: incompatible (ValueError)
preprocessor_diag: incompatible (ValueError)
preprocessor_general: incompatible dimensions (28197 vs 31496)
tree_preprocessor: incompatible dimensions (28197 vs 31496)
reduced_preprocessor: incompatible dimensions (21 vs 31496)

Decision Tree
preprocessor: incompatible (ValueError)
preprocessor_unique: incompatible (ValueError)
preprocessor_diag: incompatible (ValueError)
preprocessor_general: incompatible dimensions (28197 vs 31496)
tree_preprocessor: incompatible dimensions (28197 vs 31496)
reduced_preprocessor: incompatible dimensions (21 vs 31496)

Random Forest
preprocessor: incompatible (ValueError)
preprocessor_unique: incompatible (ValueError)
preprocessor_diag: incompatible (ValueError)
preprocessor_general: incompatible dimensions (28197 vs 31496)
tree_preprocessor: incompatible dimensions (28197 vs 

In [78]:
import numpy as np
from scipy.sparse import issparse

print("OBJECTS WITH 31,496 FEATURES")
print("----------------------------")

for name, obj in list(globals().items()):
    try:
        shape = getattr(obj, "shape", None)

        if (
            isinstance(shape, tuple)
            and len(shape) == 2
            and shape[1] == 31496
        ):
            print(
                f"{name}: shape={shape}, "
                f"type={type(obj).__name__}"
            )

    except Exception:
        continue

OBJECTS WITH 31,496 FEATURES
----------------------------
X_train_held_processed: shape=(150724, 31496), type=csr_matrix
X_test_held_processed: shape=(7076, 31496), type=csr_matrix
X_train_lr_processed: shape=(150724, 31496), type=csr_matrix
X_test_lr_processed: shape=(7076, 31496), type=csr_matrix
X_train_rf_processed: shape=(150724, 31496), type=csr_matrix
X_test_rf_processed: shape=(7076, 31496), type=csr_matrix


In [79]:
print("ORIGINAL TRANSFORMED MATRICES")
print("-----------------------------")

for name in [
    "X_train_held_processed",
    "X_test_held_processed",
    "X_train_lr_processed",
    "X_test_lr_processed",
    "X_train_rf_processed",
    "X_test_rf_processed"
]:
    matrix = globals()[name]

    print(
        f"{name}: "
        f"shape={matrix.shape}, "
        f"format={matrix.format}, "
        f"dtype={matrix.dtype}, "
        f"non-zero values={matrix.nnz:,}"
    )

ORIGINAL TRANSFORMED MATRICES
-----------------------------
X_train_held_processed: shape=(150724, 31496), format=csr, dtype=float64, non-zero values=946,122
X_test_held_processed: shape=(7076, 31496), format=csr, dtype=float64, non-zero values=44,051
X_train_lr_processed: shape=(150724, 31496), format=csr, dtype=float64, non-zero values=946,122
X_test_lr_processed: shape=(7076, 31496), format=csr, dtype=float64, non-zero values=44,051
X_train_rf_processed: shape=(150724, 31496), format=csr, dtype=float64, non-zero values=946,122
X_test_rf_processed: shape=(7076, 31496), format=csr, dtype=float64, non-zero values=44,051


In [80]:
print("FITTED PREPROCESSOR ENCODER INSPECTION")
print("--------------------------------------")

for name in [
    "preprocessor",
    "preprocessor_unique",
    "preprocessor_diag",
    "preprocessor_general",
    "tree_preprocessor",
    "reduced_preprocessor"
]:
    prep = globals()[name]

    print(f"\n{name}")

    try:
        cat_pipeline = prep.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["encoder"]

        print("Categorical input features:", len(encoder.categories_))
        print(
            "Encoded categorical dimensions:",
            sum(len(categories) for categories in encoder.categories_)
        )
        print("Encoder output:", encoder.transform(
            X_test_held[encoder.feature_names_in_]
        ).shape)

    except Exception as error:
        print("Inspection error:", type(error).__name__, str(error))

FITTED PREPROCESSOR ENCODER INSPECTION
--------------------------------------

preprocessor
Categorical input features: 9
Encoded categorical dimensions: 28209
Inspection error: AttributeError 'OneHotEncoder' object has no attribute 'feature_names_in_'

preprocessor_unique
Categorical input features: 9
Encoded categorical dimensions: 28209
Inspection error: AttributeError 'OneHotEncoder' object has no attribute 'feature_names_in_'

preprocessor_diag
Categorical input features: 2
Encoded categorical dimensions: 28174
Inspection error: AttributeError 'OneHotEncoder' object has no attribute 'feature_names_in_'

preprocessor_general
Categorical input features: 2
Encoded categorical dimensions: 28174
Inspection error: AttributeError 'OneHotEncoder' object has no attribute 'feature_names_in_'

tree_preprocessor
Categorical input features: 2
Encoded categorical dimensions: 28174
Inspection error: AttributeError 'OneHotEncoder' object has no attribute 'feature_names_in_'

reduced_preprocessor


In [81]:
print("ORIGINAL FEATURE CONFIGURATIONS")
print("--------------------------------")

for name in [
    "preprocessor",
    "preprocessor_unique",
    "preprocessor_diag",
    "preprocessor_general",
    "tree_preprocessor",
    "reduced_preprocessor"
]:
    prep = globals()[name]

    print(f"\n{name}")

    for transformer_name, transformer, columns in prep.transformers:
        if transformer_name in ["num", "cat"]:
            print(
                f"{transformer_name}: "
                f"{len(columns)} input features"
            )
            print(list(columns))

ORIGINAL FEATURE CONFIGURATIONS
--------------------------------

preprocessor
num: 31 input features
['arp.opcode', 'arp.hw.size', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'http.content_length', 'http.response', 'tcp.ack', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.seq', 'udp.port', 'udp.time_delta', 'dns.qry.name', 'dns.qry.qu', 'dns.retransmission', 'dns.retransmit_request', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver']
cat: 9 input features
['arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.request.method', 'http.request.version', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic']

preprocessor_unique
num: 31 input features
['arp.opcode', 'arp.hw.size', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'http.content_length'

In [82]:
search_terms = [
    "X_train_held_processed",
    "X_train_lr_processed",
    "X_train_rf_processed",
    "held_tree_model.fit",
    "lr_model.fit",
    "rf_model.fit"
]

print("RELEVANT NOTEBOOK EXECUTION HISTORY")
print("-----------------------------------")

for i, cell in enumerate(In):
    if any(term in cell for term in search_terms):
        print(f"\n--- CELL In[{i}] ---")
        print(cell[:5000])

RELEVANT NOTEBOOK EXECUTION HISTORY
-----------------------------------

--- CELL In[56] ---
# Prepare training and testing data
X_train_held = X_general.loc[train_indices]
X_test_held = X_general.loc[test_indices]

y_train_held = y_binary.loc[train_indices]
y_test_held = y_binary.loc[test_indices]

# Fit preprocessing using training data only
held_preprocessor = clone(tree_preprocessor)

X_train_held_processed = held_preprocessor.fit_transform(
    X_train_held
)

X_test_held_processed = held_preprocessor.transform(
    X_test_held
)

# Initialise Decision Tree
held_tree_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

held_tree_model.fit(
    X_train_held_processed,
    y_train_held
)

held_training_time = time.perf_counter() - start_time

# Measure prediction time
start_time = time.perf_counter()

y_pred_held = held_tree_model.predict(
    X_test_held_processed
)

held_pred

In [84]:
print("HISTORY OF PREPROCESSOR DEFINITIONS")
print("-----------------------------------")

for i, cell in enumerate(In):
    lines = cell.splitlines()

    matches = [
        line for line in lines
        if "tree_preprocessor =" in line
        or "X_general =" in line
    ]

    if matches:
        print(f"\n--- CELL In[{i}] ---")
        for line in lines:
            if (
                "tree_preprocessor" in line
                or "X_general" in line
                or "numerical_features" in line
                or "categorical_features" in line
            ):
                print(line)

HISTORY OF PREPROCESSOR DEFINITIONS
-----------------------------------

--- CELL In[45] ---
X_general = X_diagnostic.drop(columns=protocol_indicators)
    X_general,

--- CELL In[47] ---
X_tree = X_general.copy()
    col for col in numerical_features
    col for col in categorical_features
tree_preprocessor = ColumnTransformer(
X_train_tree_processed = tree_preprocessor.fit_transform(X_train_tree)
X_test_tree_processed = tree_preprocessor.transform(X_test_tree)

--- CELL In[83] ---
        if "tree_preprocessor =" in line
        or "X_general =" in line
                "tree_preprocessor" in line
                or "X_general" in line
                or "numerical_features" in line
                or "categorical_features" in line

--- CELL In[84] ---
        if "tree_preprocessor =" in line
        or "X_general =" in line
                "tree_preprocessor" in line
                or "X_general" in line
                or "numerical_features" in line
                or "categorical

In [85]:
for i, cell in enumerate(In):
    if "ColumnTransformer(" in cell and "tree_preprocessor" in cell:
        print("Matching cell:", i)
        print(cell)

Matching cell: 47
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import time

# Use the general feature set
X_tree = X_general.copy()

# Identify numerical and categorical columns
tree_numerical = [
    col for col in numerical_features
    if col in X_tree.columns
]

tree_categorical = [
    col for col in categorical_features
    if col in X_tree.columns
]

# Preprocess numerical features
tree_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Preprocess categorical features
tree_cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        lambda data: data.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", 

In [86]:
for i in [45, 56, 57, 58, 60, 61]:
    print(f"\n{'='*25} CELL In[{i}] {'='*25}")
    print(In[i])


========================= CELL In[45] =========================
# Diagnostic experiment: remove selected protocol-specific indicators

protocol_indicators = [
    "icmp.transmit_timestamp",
    "http.content_length",
    "http.response",
    "tcp.connection.synack",
    "tcp.connection.fin",
    "tcp.connection.rst",
    "dns.retransmission",
    "dns.retransmit_request"
]

X_general = X_diagnostic.drop(columns=protocol_indicators)

# Split data
X_train_general, X_test_general, y_train_general, y_test_general = train_test_split(
    X_general,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Identify feature types
numerical_general = X_train_general.select_dtypes(include=["number"]).columns.tolist()
categorical_general = X_train_general.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
preprocessor_general = ColumnTransformer(transformers=[
    ("num", numerical_transformer_diag, numerical_general),
    ("cat", categorical_transformer

In [87]:
print("FEATURE SELECTION COMPARISON")
print("----------------------------")

print("X_general shape:", X_general.shape)

print("\nOriginal feature lists used by In[47]:")
print("tree_numerical:", len(tree_numerical))
print("tree_categorical:", len(tree_categorical))
print("Total:", len(tree_numerical) + len(tree_categorical))

print("\nCurrent feature lists derived from X_general:")
print("Numerical:", len(numerical_features_general))
print("Categorical:", len(categorical_features_general))
print("Total:", len(numerical_features_general) + len(categorical_features_general))

print("\nOriginal numerical columns absent from current numerical list:")
print(sorted(set(tree_numerical) - set(numerical_features_general)))

print("\nOriginal categorical columns absent from current categorical list:")
print(sorted(set(tree_categorical) - set(categorical_features_general)))

print("\nColumns classified differently:")
print("Original numerical, currently categorical:",
      sorted(set(tree_numerical) & set(categorical_features_general)))
print("Original categorical, currently numerical:",
      sorted(set(tree_categorical) & set(numerical_features_general)))

print("\nOriginal held-out transformed matrix:")
print(X_train_held_processed.shape)

print("\nCurrent tree_preprocessor output:")
print(tree_preprocessor.transform(X_train_held.iloc[:5]).shape)

FEATURE SELECTION COMPARISON
----------------------------
X_general shape: (157800, 25)

Original feature lists used by In[47]:
tree_numerical: 23
tree_categorical: 2
Total: 25

Current feature lists derived from X_general:
Numerical: 23
Categorical: 2
Total: 25

Original numerical columns absent from current numerical list:
[]

Original categorical columns absent from current categorical list:
[]

Columns classified differently:
Original numerical, currently categorical: []
Original categorical, currently numerical: []

Original held-out transformed matrix:
(150724, 31496)

Current tree_preprocessor output:
(5, 28197)


In [88]:
from sklearn.compose import ColumnTransformer

print("AVAILABLE FITTED PREPROCESSORS")
print("------------------------------")

for name, obj in globals().copy().items():
    if isinstance(obj, ColumnTransformer):
        try:
            output_features = obj.transform(
                X_train_held.iloc[:1]
            ).shape[1]

            print(
                f"{name}: "
                f"{output_features} output features"
            )

        except Exception as e:
            print(f"{name}: not compatible ({type(e).__name__})")

AVAILABLE FITTED PREPROCESSORS
------------------------------
_: 31496 output features
preprocessor: not compatible (ValueError)
preprocessor_unique: not compatible (ValueError)
preprocessor_diag: not compatible (ValueError)
preprocessor_general: 28197 output features
tree_preprocessor: 28197 output features
reduced_preprocessor: 21 output features
held_preprocessor: 31496 output features
lr_preprocessor: 31496 output features
rf_preprocessor: 31496 output features
serializable_preprocessor: not compatible (NotFittedError)
_70: 31496 output features
prep: 21 output features


In [89]:
import numpy as np
from scipy import sparse

def compare_matrices(reconstructed, original):
    if reconstructed.shape != original.shape:
        return False, reconstructed.shape, original.shape

    if sparse.issparse(reconstructed) and sparse.issparse(original):
        difference = reconstructed - original
        return difference.nnz == 0, reconstructed.shape, original.shape

    return np.array_equal(reconstructed, original), reconstructed.shape, original.shape


experiments = [
    (
        "Decision Tree",
        held_preprocessor,
        X_train_held,
        X_test_held,
        X_train_held_processed,
        X_test_held_processed,
        held_tree_model,
        y_pred_held
    ),
    (
        "Logistic Regression",
        lr_preprocessor,
        X_train_lr,
        X_test_lr,
        X_train_lr_processed,
        X_test_lr_processed,
        lr_model,
        y_pred_lr
    ),
    (
        "Random Forest",
        rf_preprocessor,
        X_train_rf,
        X_test_rf,
        X_train_rf_processed,
        X_test_rf_processed,
        rf_model,
        y_pred_rf
    )
]

for name, prep, X_train, X_test, original_train, original_test, model, original_predictions in experiments:

    reconstructed_train = prep.transform(X_train)
    reconstructed_test = prep.transform(X_test)

    train_match, _, _ = compare_matrices(
        reconstructed_train, original_train
    )

    test_match, _, _ = compare_matrices(
        reconstructed_test, original_test
    )

    predictions_match = np.array_equal(
        model.predict(reconstructed_test),
        original_predictions
    )

    print(f"\n{name}")
    print("-" * len(name))
    print("Training matrix matches:", train_match)
    print("Testing matrix matches:", test_match)
    print("Predictions match:", predictions_match)
    print("Reconstructed feature count:", reconstructed_test.shape[1])


Decision Tree
-------------
Training matrix matches: False
Testing matrix matches: False
Predictions match: False
Reconstructed feature count: 31496

Logistic Regression
-------------------
Training matrix matches: False
Testing matrix matches: False
Predictions match: False
Reconstructed feature count: 31496

Random Forest
-------------
Training matrix matches: False
Testing matrix matches: False
Predictions match: False
Reconstructed feature count: 31496


In [90]:
from scipy import sparse
import numpy as np

experiments = [
    ("Decision Tree", held_preprocessor, X_train_held,
     X_train_held_processed, X_test_held, X_test_held_processed),

    ("Logistic Regression", lr_preprocessor, X_train_lr,
     X_train_lr_processed, X_test_lr, X_test_lr_processed),

    ("Random Forest", rf_preprocessor, X_train_rf,
     X_train_rf_processed, X_test_rf, X_test_rf_processed)
]

for name, prep, X_train, original_train, X_test, original_test in experiments:

    reconstructed_train = prep.transform(X_train)
    reconstructed_test = prep.transform(X_test)

    print(f"\n{name}")
    print("=" * len(name))

    for label, reconstructed, original in [
        ("Training", reconstructed_train, original_train),
        ("Testing", reconstructed_test, original_test)
    ]:
        reconstructed = sparse.csr_matrix(reconstructed)
        original = sparse.csr_matrix(original)

        difference = reconstructed - original

        max_difference = (
            np.max(np.abs(difference.data))
            if difference.nnz > 0 else 0.0
        )

        print(f"\n{label}:")
        print("Maximum absolute difference:", max_difference)
        print("Different non-zero entries:", difference.nnz)
        print("Reconstructed non-zero entries:", reconstructed.nnz)
        print("Original non-zero entries:", original.nnz)


Decision Tree

Training:
Maximum absolute difference: 2147333284.460408
Different non-zero entries: 3466652
Reconstructed non-zero entries: 3768100
Original non-zero entries: 946122

Testing:
Maximum absolute difference: 207612041.5808943
Different non-zero entries: 162748
Reconstructed non-zero entries: 174699
Original non-zero entries: 44051

Logistic Regression

Training:
Maximum absolute difference: 2147333284.460408
Different non-zero entries: 3466652
Reconstructed non-zero entries: 3768100
Original non-zero entries: 946122

Testing:
Maximum absolute difference: 207612041.5808943
Different non-zero entries: 162748
Reconstructed non-zero entries: 174699
Original non-zero entries: 44051

Random Forest

Training:
Maximum absolute difference: 2147333284.460408
Different non-zero entries: 3466652
Reconstructed non-zero entries: 3768100
Original non-zero entries: 946122

Testing:
Maximum absolute difference: 207612041.5808943
Different non-zero entries: 162748
Reconstructed non-zero en

In [91]:
def inspect_preprocessor(name, preprocessor):
    print(f"\n{name}")
    print("-" * len(name))

    for transformer_name, transformer, columns in preprocessor.transformers_:
        print(f"\nTransformer: {transformer_name}")
        print("Columns:", len(columns))

        if hasattr(transformer, "steps"):
            for step_name, step_object in transformer.steps:
                print(f"  {step_name}: {step_object}")
        else:
            print("  ", transformer)


inspect_preprocessor("Original tree_preprocessor", tree_preprocessor)
inspect_preprocessor("Surviving held_preprocessor", held_preprocessor)


Original tree_preprocessor
--------------------------

Transformer: num
Columns: 23
  imputer: SimpleImputer(strategy='median')

Transformer: cat
Columns: 2
  imputer: SimpleImputer(strategy='most_frequent')
  to_string: FunctionTransformer(feature_names_out='one-to-one',
                    func=<function <lambda> at 0x171f336a0>)
  encoder: OneHotEncoder(handle_unknown='ignore')

Surviving held_preprocessor
---------------------------

Transformer: num
Columns: 23
  imputer: SimpleImputer(strategy='median')
  scaler: StandardScaler()

Transformer: cat
Columns: 2
  imputer: SimpleImputer(strategy='most_frequent')
  to_string: FunctionTransformer(feature_names_out='one-to-one',
                    func=<function convert_to_string at 0x1757b63e0>)
  encoder: OneHotEncoder(handle_unknown='ignore')


In [92]:
print("ORIGINAL NUMERICAL PIPELINE")
print(tree_preprocessor.named_transformers_["num"])

print("\nSURVIVING HELD-OUT NUMERICAL PIPELINE")
print(held_preprocessor.named_transformers_["num"])

ORIGINAL NUMERICAL PIPELINE
Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

SURVIVING HELD-OUT NUMERICAL PIPELINE
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [93]:
terms = [
    "held_preprocessor =",
    "tree_preprocessor =",
    "numerical_transformer_diag ="
]

for i, cell in enumerate(In):
    if any(term in cell for term in terms):
        print(f"\n--- CELL In[{i}] ---")
        print(cell)


--- CELL In[43] ---
# Diagnostic experiment: exclude strongly class-associated categorical fields

suspicious_categorical = [
    "arp.dst.proto_ipv4",
    "arp.src.proto_ipv4",
    "http.request.method",
    "http.request.version",
    "mqtt.conack.flags",
    "mqtt.protoname",
    "mqtt.topic"
]

X_diagnostic = X_reduced.drop(columns=suspicious_categorical)

# Recreate the train/test split using the same random seed
X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(
    X_diagnostic,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Identify remaining feature types
numerical_diag = X_train_diag.select_dtypes(include=["number"]).columns.tolist()
categorical_diag = X_train_diag.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
numerical_transformer_diag = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_diag = Pipeline(steps=[
    ("im

In [94]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from scipy import sparse
import numpy as np

# Named function replaces the original anonymous lambda
def convert_to_string_verified(data):
    return data.astype(str)


# Reconstruct the original numerical pipeline
verified_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Reconstruct the original categorical pipeline
verified_cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_string", FunctionTransformer(
        convert_to_string_verified,
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Reconstruct the original ColumnTransformer
verified_preprocessor = ColumnTransformer(
    transformers=[
        ("num", verified_num_pipeline, tree_numerical),
        ("cat", verified_cat_pipeline, tree_categorical)
    ]
)

# Fit only on the held-out training partition
X_train_verified = verified_preprocessor.fit_transform(X_train_held)
X_test_verified = verified_preprocessor.transform(X_test_held)

# Compare against the original saved matrices
train_difference = sparse.csr_matrix(
    X_train_verified - X_train_held_processed
)

test_difference = sparse.csr_matrix(
    X_test_verified - X_test_held_processed
)

print("RECONSTRUCTED PREPROCESSOR VERIFICATION")
print("---------------------------------------")
print("Training dimensions:", X_train_verified.shape)
print("Testing dimensions:", X_test_verified.shape)

print("\nTraining matrix matches:",
      train_difference.nnz == 0)

print("Testing matrix matches:",
      test_difference.nnz == 0)

print("\nMaximum training difference:",
      np.max(np.abs(train_difference.data))
      if train_difference.nnz else 0.0)

print("Maximum testing difference:",
      np.max(np.abs(test_difference.data))
      if test_difference.nnz else 0.0)

RECONSTRUCTED PREPROCESSOR VERIFICATION
---------------------------------------
Training dimensions: (150724, 31496)
Testing dimensions: (7076, 31496)

Training matrix matches: True
Testing matrix matches: True

Maximum training difference: 0.0
Maximum testing difference: 0.0


In [95]:
# Verify that reconstructed preprocessing reproduces original predictions

verified_tree_predictions = held_tree_model.predict(X_test_verified)
verified_lr_predictions = lr_model.predict(X_test_verified)
verified_rf_predictions = rf_model.predict(X_test_verified)

prediction_checks = {
    "Decision Tree": np.array_equal(
        verified_tree_predictions, np.asarray(y_pred_held)
    ),
    "Logistic Regression": np.array_equal(
        verified_lr_predictions, np.asarray(y_pred_lr)
    ),
    "Random Forest": np.array_equal(
        verified_rf_predictions, np.asarray(y_pred_rf)
    )
}

print("PREDICTION EQUIVALENCE VERIFICATION")
print("-----------------------------------")

for model_name, matches in prediction_checks.items():
    print(f"{model_name}: {'MATCH' if matches else 'MISMATCH'}")

print("\nNumber of differing predictions:")
print("Decision Tree:",
      np.sum(verified_tree_predictions != np.asarray(y_pred_held)))
print("Logistic Regression:",
      np.sum(verified_lr_predictions != np.asarray(y_pred_lr)))
print("Random Forest:",
      np.sum(verified_rf_predictions != np.asarray(y_pred_rf)))

PREDICTION EQUIVALENCE VERIFICATION
-----------------------------------
Decision Tree: MATCH
Logistic Regression: MATCH
Random Forest: MATCH

Number of differing predictions:
Decision Tree: 0
Logistic Regression: 0
Random Forest: 0


In [96]:
from copy import deepcopy
from sklearn.pipeline import Pipeline

# Assemble complete pipelines without retraining classifiers
verified_tree_pipeline = Pipeline([
    ("preprocessor", deepcopy(verified_preprocessor)),
    ("classifier", held_tree_model)
])

verified_lr_pipeline = Pipeline([
    ("preprocessor", deepcopy(verified_preprocessor)),
    ("classifier", lr_model)
])

verified_rf_pipeline = Pipeline([
    ("preprocessor", deepcopy(verified_preprocessor)),
    ("classifier", rf_model)
])

# Verify predictions directly from the raw test features
pipeline_tree_predictions = verified_tree_pipeline.predict(X_test_held)
pipeline_lr_predictions = verified_lr_pipeline.predict(X_test_held)
pipeline_rf_predictions = verified_rf_pipeline.predict(X_test_held)

print("COMPLETE PIPELINE VERIFICATION")
print("------------------------------")

print("Decision Tree:",
      np.array_equal(pipeline_tree_predictions, np.asarray(y_pred_held)))

print("Logistic Regression:",
      np.array_equal(pipeline_lr_predictions, np.asarray(y_pred_lr)))

print("Random Forest:",
      np.array_equal(pipeline_rf_predictions, np.asarray(y_pred_rf)))

COMPLETE PIPELINE VERIFICATION
------------------------------
Decision Tree: True
Logistic Regression: True
Random Forest: True


In [97]:
import os
import joblib

# Save each complete pipeline
joblib.dump(verified_tree_pipeline, "decision_tree_pipeline.joblib")
joblib.dump(verified_lr_pipeline, "logistic_regression_pipeline.joblib")
joblib.dump(verified_rf_pipeline, "random_forest_pipeline.joblib")

# Measure serialized file sizes
pipeline_files = {
    "Logistic Regression": "logistic_regression_pipeline.joblib",
    "Decision Tree": "decision_tree_pipeline.joblib",
    "Random Forest": "random_forest_pipeline.joblib"
}

print("COMPLETE PIPELINE SERIALIZED SIZES")
print("----------------------------------")

for model_name, file_path in pipeline_files.items():
    size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"{model_name}: {size_mb:.4f} MB")

COMPLETE PIPELINE SERIALIZED SIZES
----------------------------------
Logistic Regression: 2.7669 MB
Decision Tree: 2.5294 MB
Random Forest: 2.6462 MB


### Verification of Saved Model Pipelines

This stage verifies the integrity and reproducibility of the serialized machine learning pipelines. Each saved pipeline is reloaded from its `.joblib` file and used to generate predictions on the same test dataset used during the original pipeline verification. The predictions are then compared with those produced by the original pipelines. Matching predictions confirm that serialization and reloading have preserved the models' behaviour.

In [98]:
# Reload saved pipelines
loaded_lr_pipeline = joblib.load("logistic_regression_pipeline.joblib")
loaded_tree_pipeline = joblib.load("decision_tree_pipeline.joblib")
loaded_rf_pipeline = joblib.load("random_forest_pipeline.joblib")

# Generate predictions using the reloaded pipelines
loaded_lr_predictions = loaded_lr_pipeline.predict(X_test)
loaded_tree_predictions = loaded_tree_pipeline.predict(X_test)
loaded_rf_predictions = loaded_rf_pipeline.predict(X_test)

# Compare predictions with the original verified pipelines
print("SERIALIZED PIPELINE VERIFICATION")
print("--------------------------------")

print(
    "Logistic Regression:",
    (loaded_lr_predictions == verified_lr_pipeline.predict(X_test)).all()
)

print(
    "Decision Tree:",
    (loaded_tree_predictions == verified_tree_pipeline.predict(X_test)).all()
)

print(
    "Random Forest:",
    (loaded_rf_predictions == verified_rf_pipeline.predict(X_test)).all()
)

SERIALIZED PIPELINE VERIFICATION
--------------------------------
Logistic Regression: True
Decision Tree: True
Random Forest: True


### Comparative Evaluation of the Machine Learning Models

This stage consolidates the evaluation metrics of Logistic Regression, Decision Tree and Random Forest into a single comparison. Accuracy, precision, recall and F1-score are used to assess classification performance. The comparison provides a consistent basis for identifying differences in detection effectiveness across the models.

### Comparative Performance Evaluation

The three verified pipelines are evaluated using the same test dataset and target labels. Accuracy, precision, recall and F1-score are calculated for each model and presented in a consolidated table. This comparison helps identify the model that provides the strongest classification performance under the current evaluation setup.

In [100]:
# Inspect existing test datasets and target variables

print("AVAILABLE TEST VARIABLES")
print("------------------------")

for name, value in list(globals().items()):
    if name.startswith(("X_test", "y_test")):
        if hasattr(value, "shape"):
            print(f"{name}: {value.shape}")
        elif hasattr(value, "__len__"):
            print(f"{name}: length = {len(value)}")

AVAILABLE TEST VARIABLES
------------------------
X_test: (7076, 25)
y_test: (31560,)
X_test_processed: (31560, 28240)
X_test_unique_processed: (31560, 28240)
X_test_diag: (31560, 33)
y_test_diag: (31560,)
X_test_diag_processed: (31560, 28205)
X_test_general: (31560, 25)
y_test_general: (31560,)
X_test_general_processed: (31560, 28197)
X_test_num: (31560, 23)
y_test_num: (31560,)
X_test_num_processed: (31560, 23)
X_test_tree: (31560, 25)
y_test_tree: (31560,)
X_test_tree_processed: (31560, 28197)
X_test_reduced: (31560, 21)
y_test_reduced: (31560,)
X_test_reduced_processed: (31560, 21)
X_test_held: (7076, 25)
y_test_held: (7076,)
X_test_held_processed: (7076, 31496)
X_test_lr: (7076, 25)
y_test_lr: (7076,)
X_test_lr_processed: (7076, 31496)
X_test_rf: (7076, 25)
y_test_rf: (7076,)
X_test_rf_processed: (7076, 31496)
X_test_verified: (7076, 31496)


In [101]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the labels corresponding to the held-out test dataset
y_true = y_test_held

# Store predictions from the reloaded pipelines
models = {
    "Logistic Regression": loaded_lr_predictions,
    "Decision Tree": loaded_tree_predictions,
    "Random Forest": loaded_rf_predictions
}

# Calculate evaluation metrics
comparison_results = []

for model_name, predictions in models.items():
    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0)
    })

# Display comparative results
performance_comparison = pd.DataFrame(comparison_results)

print("HELD-OUT MODEL PERFORMANCE COMPARISON")
print("-------------------------------------")

display(
    performance_comparison.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)

HELD-OUT MODEL PERFORMANCE COMPARISON
-------------------------------------


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.7771,0.5945,0.9056,0.7178
1,Decision Tree,0.9672,0.9975,0.8975,0.9449
2,Random Forest,0.8185,0.6529,0.8975,0.7559


### Confusion Matrix Analysis

Confusion matrices are generated to examine the classification outcomes of each model on the held-out test dataset. They show true positives, true negatives, false positives and false negatives, providing a more detailed assessment of how effectively each model distinguishes normal traffic from malicious traffic.